In [1]:
!pip install pandas python-dotenv groq


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import os
from dotenv import load_dotenv
from groq import Groq
from IPython.display import display

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found in .env")

client = Groq(api_key=api_key)

In [3]:
# Load the dataset (Task - 1)
df = pd.read_csv("patient_data.csv")


print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (100, 18)


,patient_id,patient_name,age,gender,diagnosis,current_medication,lab_test,lab_value,lab_unit,last_visit_date,next_scheduled_visit,days_since_last_visit,missed_last_appointment,vitals_bp_systolic,vitals_bp_diastolic,vitals_heart_rate,vitals_spo2,notes
0,P0001,Aarav Sharma,68,Male,Type 2 Diabetes,Metformin 500mg,HbA1c,9.9,%,2024-04-24,2024-06-08,251,No,174,70,95,96,Patient reports good medication adherence.
1,P0002,Priya Patel,29,Male,Heart Failure,Furosemide 40mg,BNP,317.6,pg/mL,2024-01-14,2024-02-28,352,No,174,91,72,97,Patient reports no new symptoms.
2,P0003,Rohan Mehta,45,Male,"Type 2 Diabetes, CKD","Insulin Glargine 10U, Losartan 50mg",HbA1c,11.4,%,2024-08-04,2024-10-03,149,No,132,86,64,91,Medication dose adjusted this visit.
3,P0004,Anjali Singh,34,Female,"COPD, Heart Failure","Tiotropium 18mcg, Furosemide 40mg",FEV1%,33.8,%,2024-01-23,2024-04-22,343,No,153,70,93,94,Patient reports no new symptoms.
4,P0005,Vikram Nair,51,Male,"Hypertension, Heart Failure","Amlodipine 5mg, Furosemide 40mg",BP Systolic,124.9,mmHg,2024-04-26,2024-06-25,249,No,134,71,82,94,Patient education provided on diet and lifestyle.


In [4]:
df.describe()

,age,lab_value,days_since_last_visit,vitals_bp_systolic,vitals_bp_diastolic,vitals_heart_rate,vitals_spo2
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,55.360000,150.810000,211.390000,145.570000,87.680000,82.420000,94.500000
std,15.251773,279.918492,87.498675,20.375889,13.625496,14.460102,2.794294
min,28.000000,1.700000,65.000000,105.000000,65.000000,58.000000,90.000000
25%,44.000000,9.675000,132.500000,130.000000,75.750000,72.000000,92.000000
50%,55.000000,34.950000,219.000000,145.500000,87.500000,83.000000,95.000000
75%,68.000000,144.600000,287.750000,161.250000,99.250000,95.000000,97.000000
max,82.000000,1164.000000,360.000000,181.000000,110.000000,108.000000,99.000000


In [5]:
df.isnull().sum()   

patient_id                 0
patient_name               0
age                        0
gender                     0
diagnosis                  0
current_medication         0
lab_test                   0
lab_value                  0
lab_unit                   0
last_visit_date            0
next_scheduled_visit       0
days_since_last_visit      0
missed_last_appointment    0
vitals_bp_systolic         0
vitals_bp_diastolic        0
vitals_heart_rate          0
vitals_spo2                0
notes                      0
dtype: int64

In [6]:
def call_llm(prompt, retries=2):
    models = ["llama-3.1-8b-instant", "llama-3.3-70b-versatile"]

    for model in models:
        for attempt in range(retries):
            try:
                res = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.3
                )
                return res.choices[0].message.content
            except Exception as e:
                print(f"[Retry {attempt+1}] {model} failed")
                time.sleep(1)

    return "LLM ERROR"

In [7]:
# Define Agents Tools (Task - 2)
def risk_tool(patient):
    prompt = f"""
    You are a clinical AI.

    Classify risk: LOW, MEDIUM, HIGH.

    Consider:
    vitals, labs, diagnosis, missed visits.

    Patient:
    {patient}

    Output:
    Risk + short reason
    """
    return call_llm(prompt)

In [8]:
def action_tool(patient, risk):
    prompt = f"""
    Create a follow-up plan.

    Risk: {risk}

    Include:
    urgency, actions, timeline.

    Patient:
    {patient}
    """
    return call_llm(prompt)

In [9]:
def missed_tool(patient):
    prompt = f"""
    Patient missed appointment.

    Provide:
    outreach, risk, next steps.

    Patient:
    {patient}
    """
    return call_llm(prompt)

In [10]:
# Agent Decision (Task -3)
def agent_decision(patient):
    prompt = f"""
    You are an AI agent.

    Tools:
    - risk_tool
    - action_tool
    - missed_tool

    Patient:
    {patient}

    Decide best tool.

    Output:
    TOOL: <name>
    """
    return call_llm(prompt).lower()

In [11]:
def run_agent(patient):
    decision = agent_decision(patient)

    if "risk_tool" in decision:
        risk = risk_tool(patient)
        plan = action_tool(patient, risk)
        return {"risk": risk, "plan": plan}

    elif "missed_tool" in decision:
        follow = missed_tool(patient)
        return {"followup": follow}

    else:
        return {"message": "No action"}

In [12]:
def agent_decision(patient):
    prompt = f"""
    You are an AI agent.

    Tools:
    - risk_tool
    - action_tool
    - missed_tool

    Patient:
    {patient}

    Decide best tool.

    Output:
    TOOL: <name>
    """
    return call_llm(prompt).lower()

In [13]:
def run_agent(patient):
    decision = agent_decision(patient)

    if "risk_tool" in decision:
        risk = risk_tool(patient)
        plan = action_tool(patient, risk)
        return {"risk": risk, "plan": plan}

    elif "missed_tool" in decision:
        follow = missed_tool(patient)
        return {"followup": follow}

    else:
        return {"message": "No action"}

In [14]:
#Single Patient (Task -4)
patient = df.iloc[8].to_dict()

result = run_agent(patient)

print("Patient:", patient["patient_name"])
print(result)

Patient: Amit Joshi
{'risk': "Based on the provided patient data, I would classify the risk as:\n\n**MEDIUM**\n\nReason: Elevated blood pressure (BP Systolic: 159.3 mmHg) and recent missed appointment (66 days since last visit) indicate a moderate level of risk. The patient's diagnosis of Hypertension and Heart Failure also contributes to this classification. However, the patient's current medication and lab values suggest that their condition is being managed, but more frequent monitoring is necessary to prevent complications.\n\nNote: A more detailed analysis would be required to provide a more accurate risk assessment, but based on the provided data, a medium risk classification seems appropriate.", 'plan': "**Follow-up Plan for Patient P0009 (Amit Joshi)**\n\n**Risk Classification:** Medium\n\n**Reason:** Elevated blood pressure (BP Systolic: 159.3 mmHg) and recent missed appointment (66 days since last visit) indicate a moderate level of risk. The patient's diagnosis of Hypertensi

In [15]:
def format_output(patient, result):
    print("\n" + "="*60)
    print(f"Patient: {patient['patient_name']}")
    print("="*60)

    print("\n Risk Assessment")
    print("-"*60)
    print(result['risk'])

    print("\n Follow-up Plan")
    print("-"*60)
    print(result['plan'])
    print("="*60)


format_output(patient, result)


Patient: Amit Joshi

 Risk Assessment
------------------------------------------------------------
Based on the provided patient data, I would classify the risk as:

**MEDIUM**

Reason: Elevated blood pressure (BP Systolic: 159.3 mmHg) and recent missed appointment (66 days since last visit) indicate a moderate level of risk. The patient's diagnosis of Hypertension and Heart Failure also contributes to this classification. However, the patient's current medication and lab values suggest that their condition is being managed, but more frequent monitoring is necessary to prevent complications.

Note: A more detailed analysis would be required to provide a more accurate risk assessment, but based on the provided data, a medium risk classification seems appropriate.

 Follow-up Plan
------------------------------------------------------------
**Follow-up Plan for Patient P0009 (Amit Joshi)**

**Risk Classification:** Medium

**Reason:** Elevated blood pressure (BP Systolic: 159.3 mmHg) an

In [16]:
# Missed Appointments (Task -5)
missed_df = df[df["missed_last_appointment"] == "Yes"]

outputs = []

for _, row in missed_df.iterrows():
    patient = row.to_dict()
    result = run_agent(patient)

    outputs.append({
        "patient_id": patient["patient_id"],
        "result": result
    })

print("Processed:", len(outputs))
print(outputs[:2])

Processed: 15
[{'patient_id': 'P0018', 'result': {'followup': "**Outreach Plan:**\n\n1. **Phone Call:** Reach out to the patient via phone to remind them of the missed appointment and inquire about their current health status. This will help in understanding the reason for missing the appointment and provide an opportunity to reschedule.\n2. **Email or SMS:** Send a follow-up email or SMS to the patient to confirm the missed appointment and provide a link to reschedule online.\n3. **Personalized Message:** Leave a personalized voicemail or message for the patient, addressing them by name and expressing concern about their missed appointment.\n\n**Risk Assessment:**\n\nBased on the patient's medical history and current vitals, the following risks are identified:\n\n1. **Uncontrolled Blood Pressure:** The patient's systolic blood pressure (160 mmHg) is higher than the recommended level, which may increase the risk of cardiovascular events.\n2. **Increased Heart Rate:** The patient's hear

In [17]:
import re

def clean_text(text):
    """Remove markdown symbols and extra spaces"""
    if not text:
        return ""
    text = re.sub(r"\*\*|__|###|\*|`", "", text)   # remove markdown
    text = re.sub(r"\n+", "\n", text)              # normalize newlines
    return text.strip()


def print_patient_summary(outputs, limit=None):
    print("\n" + "="*80)
    print(" MISSED APPOINTMENTS SUMMARY")
    print("="*80)

    for i, item in enumerate(outputs):
        if limit and i >= limit:
            break

        patient_id = item.get("patient_id", "N/A")
        result = item.get("result", {})

        print(f"\n Patient ID: {patient_id}")
        print("-"*80)

        # --- Risk ---
        risk = result.get("risk")
        if risk:
            print("\n Risk Assessment:")
            print(clean_text(risk)[:400], "...")

        # --- Plan ---
        plan = result.get("plan")
        if plan:
            print("\n📋 Care Plan:")
            print(clean_text(plan)[:400], "...")

        # --- Follow-up ---
        followup = result.get("followup")
        if followup:
            print("\n Follow-up Actions:")
            print(clean_text(followup)[:400], "...")

        print("\n" + "="*80)

print_patient_summary(outputs)


 MISSED APPOINTMENTS SUMMARY

 Patient ID: P0018
--------------------------------------------------------------------------------

 Follow-up Actions:
Outreach Plan:
1. Phone Call: Reach out to the patient via phone to remind them of the missed appointment and inquire about their current health status. This will help in understanding the reason for missing the appointment and provide an opportunity to reschedule.
2. Email or SMS: Send a follow-up email or SMS to the patient to confirm the missed appointment and provide a link to reschedule onlin ...


 Patient ID: P0031
--------------------------------------------------------------------------------

 Follow-up Actions:
Outreach Plan:
To address the missed appointment, consider the following outreach plan:
1. Phone Call: Reach out to the patient via phone to inquire about the missed appointment and express concern for their health. This will help to:
	 Re-establish communication with the patient.
	 Understand the reason for missing th